In [9]:
import os
import re
import time
import pandas as pd
from Bio import Entrez, SeqIO
from dotenv import load_dotenv

# --- 1. SETUP & ALIASES ---
load_dotenv("keys.env")
Entrez.email = os.getenv("EMAIL")
Entrez.api_key = os.getenv("NCBI_API_KEY")

GENE_DATABASE = {
    "COI": ["COI", "COX1", "COX-1", "COXI", "cytochrome c oxidase subunit 1", "cytochrome c oxidase subunit I"],
    "18S": ["18S", "18S ribosomal RNA", "18S rRNA", "SSU"],
    "ITS": ["ITS", "internal transribed spacer", "ITS1", "ITS2"]
}

def setup_folders(organism, user_gene):
    """Creates a gene-specific folder hierarchy."""
    base_path = f"../API_NCBI/{user_gene}"
    folders = [
        f"{base_path}/Sequences", 
        f"{base_path}/Metadata",
        "../raw_data/Outgroups",
        "../raw_data"
    ]
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
    return base_path

# --- 2. CORE FUNCTIONS ---

def get_gene_aliases(user_gene):
    return GENE_DATABASE.get(user_gene.upper(), [user_gene])

def fetch_and_extract(organism, user_gene):
    gene_targets = get_gene_aliases(user_gene)
    search_query = " OR ".join([f'"{g}"[All Fields]' for g in gene_targets])
    full_query = f'("{organism}"[Organism] AND ({search_query}))'
    
    print(f"🔎 Searching NCBI for: {full_query}")
    handle = Entrez.esearch(db="nucleotide", term=full_query, usehistory="y")
    search_results = Entrez.read(handle)
    count = int(search_results["Count"])
    
    fasta_path = f"../API_NCBI/Sequences/{organism}_{user_gene}_raw.fasta"
    meta_path = f"../API_NCBI/Metadata/{organism}_{user_gene}_NCBI_info.tsv"
        
    metadata_rows = []

    with open(fasta_path, "w") as f_out:
        batch_size = 20
        for start in range(0, count, batch_size):
            try:
                handle = Entrez.efetch(
                    db="nucleotide", retstart=start, retmax=batch_size,
                    webenv=search_results["WebEnv"], query_key=search_results["QueryKey"],
                    rettype="gb", retmode="text"
                )
                records = SeqIO.parse(handle, "genbank")
                
                for record in records:
                    # --- 1. Extract Country & Paper Metadata ---
                    country = "Unknown"
                    isolation_source = "Unknown"

                    for feat in record.features:
                        if feat.type == 'source':
                            country = feat.qualifiers.get('country', feat.qualifiers.get('geo_loc_name', ['Unknown']))[0]
                            isolation_source = feat.qualifiers.get('isolation_source', ['Unknown'])
                    
                    paper_title = "No Title"
                    journal = "Unknown"
                    authors = "Unknown"

                    if record.annotations.get('references'):
                        paper_title = record.annotations['references'][0].title or "No Title"
                        ref = record.annotations['references'][0] # Grabs the first reference
                        paper_title = ref.title or "No Title"
                        journal = ref.journal or "Unknown"
                        authors = ref.authors or "Unknown"

                    full_title = record.description

                    metadata_rows.append({
                        "Sequence_ID": record.id,
                        "Full_Description": full_title,
                        "Species": record.annotations.get('organism', 'Unknown'),
                        "Country": country.split(':')[0], # Clean "Greece: Aegean" to "Greece"
                        "Paper_Title": paper_title,
                        "Isolation_Source": isolation_source [0],
                        "Authors": authors,
                        "Journal": journal,
                        "Paper_Title": paper_title
                    })

              # --- 2. Extract Sequence (with Genome Logic) ---
                    final_seq = record.seq
                    final_title = record.description
                    
                    if "complete genome" in record.description.lower():
                        gene_found = False
                        for feature in record.features:
                            if feature.type in ["CDS", "gene"]:
                                # We check the 'gene' and 'product' tags for synonyms
                                gene_tag = str(feature.qualifiers.get("gene", "")).lower()
                                prod_tag = str(feature.qualifiers.get("product", "")).lower()
                                
                                if any(t.lower() in gene_tag or t.lower() in prod_tag for t in gene_targets):
                                    # SUCCESS: Extract only the gene part
                                    final_seq = feature.location.extract(record.seq)
                                    
                                    # CLEAN TITLE: Remove "complete genome", add the actual Gene Name
                                    clean_desc = re.sub(r",? complete genome.*", "", record.description, flags=re.IGNORECASE)
                                    final_title = f"{clean_desc}, {user_gene}"
                                    
                                    gene_found = True
                                    break # Stop looking once we find the first match
                        
                        if not gene_found:
                            print(f"⚠️ Warning: {user_gene} not found in genome {record.id}. Skipping.")
                            continue # Don't save the whole genome if we didn't find the gene!
                    
                    # Write result (either the original barcode or the extracted gene)
                    f_out.write(f">{record.id} {final_title}\n{final_seq}\n")
                
                print(f" Progress: {start+batch_size}/{count} processed...")
                time.sleep(0.3)
            except Exception as e:
                print(f"❌ Error in batch {start}: {e}")

    # Save Metadata to TSV
    pd.DataFrame(metadata_rows).to_csv(meta_path, sep='\t', index=False)
    return fasta_path

def append_local_data(ncbi_fasta_path, user_gene):
    """
    Searches '../raw_data' and '../raw_data/Outgroups' for files matching the gene
    and appends them to the NCBI results.
    """
    combined_path = ncbi_fasta_path.replace("_raw.fasta", "_combined.fasta")
    
    # 1. Load NCBI records
    all_records = list(SeqIO.parse(ncbi_fasta_path, "fasta"))
    
    # 2. Append Local Data from ../raw_data
    def append_local_data(ncbi_fasta_path, user_gene, metadata_rows):
    """
    Appends local data AND updates the metadata list to keep them in sync.
    """
    combined_path = ncbi_fasta_path.replace("_raw.fasta", "_combined.fasta")
    all_records = list(SeqIO.parse(ncbi_fasta_path, "fasta"))
    
    # Folders to check
    folders = ["../raw_data", "../raw_data/Outgroups"]
    
    for folder in folders:
        if os.path.exists(folder):
            matching_files = [
                f for f in os.listdir(folder) 
                if f.endswith((".fasta", ".fa")) and user_gene.upper() in f.upper()
            ]
            
            for f in matching_files:
                local_records = list(SeqIO.parse(os.path.join(folder, f), "fasta"))
                for record in local_records:
                    all_records.append(record)
                    
                    # --- SYNC METADATA ---
                    # Add a placeholder row so R doesn't crash or lose this sequence
                    metadata_rows.append({
                        "Sequence_ID": record.id,
                        "Full_Description": record.description,
                        "Species": "Local/Outgroup",
                        "Country": "Local",
                        "Paper_Title": "Local Data",
                        "Isolation_Source": "Unknown",
                        "Authors": "User",
                        "Journal": "Local File: " + f
                    })
                print(f"➕ Appended {len(local_records)} sequences from: {f}")

    SeqIO.write(all_records, combined_path, "fasta")
    return combined_path
    

    # 3. Append Outgroup from ../raw_data/Outgroups
    outgroup_folder = "../raw_data/Outgroups"
    if os.path.exists(outgroup_folder):
        # Look for a file that contains 'Outgroup' and the gene name
        out_files = [
            f for f in os.listdir(outgroup_folder)
            if f.endswith((".fasta", ".fa")) and "OUTGROUP" in f.upper() and user_gene.upper() in f.upper()
        ]
        
        for f in out_files:
            out_records = list(SeqIO.parse(os.path.join(outgroup_folder, f), "fasta"))
            all_records.extend(out_records)
            print(f"🎯 Outgroup added from: {f}")

             # --- SYNC METADATA ---
                    # Add a placeholder row so R doesn't crash or lose this sequence
                    metadata_rows.append({
                        "Sequence_ID": record.id,
                        "Full_Description": record.description,
                        "Species": "Local/Outgroup",
                        "Country": "Local",
                        "Paper_Title": "Local Data",
                        "Isolation_Source": "Unknown",
                        "Authors": "User",
                        "Journal": "Local File: " + f
                    })
                print(f"➕ Appended {len(out_records)} sequences from: {f}")

    SeqIO.write(all_records, combined_path, "fasta")
    return combined_path

    # 4. Save the final combined file
    SeqIO.write(all_records, combined_path, "fasta")
    print(f"\n✨ Final combined dataset saved: {combined_path} ({len(all_records)} total seqs)")
    
# --- 3. THE INTERFACE ---
if __name__ == "__main__":
    setup_folders()
    print("\n🧬 PHYLOGENETICS PIPELINE")
    org = input("Enter Organism name: ").strip()
    gene = input("Enter Gene (COI, 18S, etc.): ").strip()

    if org and gene:
        raw_fasta = fetch_and_extract(org, gene)
        append_local_data(raw_fasta, gene)
        print("\n✅ Process complete. Check the 'API_NCBI' folder!")


🧬 PHYLOGENETICS PIPELINE


Enter Organism name:  Pseudodiaptomus
Enter Gene (COI, 18S, etc.):  18S


🔎 Searching NCBI for: ("Pseudodiaptomus"[Organism] AND ("18S"[All Fields] OR "18S ribosomal RNA"[All Fields] OR "18S rRNA"[All Fields] OR "SSU"[All Fields]))
 Progress: 20/30 processed...
 Progress: 40/30 processed...
➕ Appended 3 sequences from local file: 18S_Pseudodiaptomus_marinus_Thesaloniki Bay.fasta
🎯 Outgroup added from: Outgroup_Hesperodiaptomus_18S.fasta

✨ Final combined dataset saved: ../API_NCBI/Sequences/Pseudodiaptomus_18S_combined.fasta (34 total seqs)

✅ Process complete. Check the 'API_NCBI' folder!


In [ ]:
Second

In [ ]:
import os
import re
import time
import pandas as pd
from Bio import Entrez, SeqIO
from dotenv import load_dotenv

# --- 1. SETUP ---
load_dotenv("keys.env")
Entrez.email = os.getenv("EMAIL")
Entrez.api_key = os.getenv("NCBI_API_KEY")

GENE_DATABASE = {
    "COI": ["COI", "COX1", "COX-1", "COXI", "cytochrome c oxidase subunit 1", "cytochrome c oxidase subunit I"],
    "18S": ["18S", "18S ribosomal RNA", "18S rRNA", "SSU"],
    "ITS": ["ITS", "internal transribed spacer", "ITS1", "ITS2"]
}

def setup_folders(user_gene):
    """Creates a gene-specific folder hierarchy."""
    base_path = f"../API_NCBI/{user_gene}"
    folders = [
        f"{base_path}/Sequences", 
        f"{base_path}/Metadata",
        "../raw_data/Outgroups"
    ]
    for folder in folders:
        os.makedirs(folder, exist_ok=True)
    return base_path

def get_gene_aliases(user_gene):
    return GENE_DATABASE.get(user_gene.upper(), [user_gene])

# --- 2. CORE FUNCTIONS ---

def fetch_and_extract(organism, user_gene, base_path):
    gene_targets = get_gene_aliases(user_gene)
    search_query = " OR ".join([f'"{g}"[All Fields]' for g in gene_targets])
    full_query = f'("{organism}"[Organism] AND ({search_query}))'
    
    print(f"🔎 Searching NCBI for: {full_query}")
    handle = Entrez.esearch(db="nucleotide", term=full_query, usehistory="y")
    search_results = Entrez.read(handle)
    count = int(search_results["Count"])
    
    fasta_path = f"{base_path}/Sequences/{organism}_{user_gene}_raw.fasta"
    metadata_rows = []

    with open(fasta_path, "w") as f_out:
        batch_size = 20
        for start in range(0, count, batch_size):
            try:
                handle = Entrez.efetch(
                    db="nucleotide", retstart=start, retmax=batch_size,
                    webenv=search_results["WebEnv"], query_key=search_results["QueryKey"],
                    rettype="gb", retmode="text"
                )
                records = list(SeqIO.parse(handle, "genbank"))
                
                for record in records:
                    # Default values
                    country = "Unknown"
                    isolation_source = "Unknown"
                    for feat in record.features:
                        if feat.type == 'source':
                            country = feat.qualifiers.get('country', feat.qualifiers.get('geo_loc_name', ['Unknown']))[0]
                            isolation_source = feat.qualifiers.get('isolation_source', ['Unknown'])[0]
                    
                    ref = record.annotations['references'][0] if record.annotations.get('references') else None
                    paper_title = ref.title if ref else "No Title"
                    journal = ref.journal if ref else "Unknown"
                    authors = ref.authors if ref else "Unknown"

                    final_seq = record.seq
                    final_title = record.description
                    save_this_record = True # Default for standard barcodes

                    # Genome Logic
                    if "complete genome" in record.description.lower():
                        save_this_record = False
                        for feature in record.features:
                            if feature.type in ["CDS", "gene"]:
                                gene_tag = str(feature.qualifiers.get("gene", "")).lower()
                                prod_tag = str(feature.qualifiers.get("product", "")).lower()
                                if any(t.lower() in gene_tag or t.lower() in prod_tag for t in gene_targets):
                                    final_seq = feature.location.extract(record.seq)
                                    clean_desc = re.sub(r",? complete genome.*", "", record.description, flags=re.IGNORECASE)
                                    final_title = f"{clean_desc}, {user_gene}"
                                    save_this_record = True
                                    break
                    
                    if save_this_record:
                        # WRITE TO FASTA
                        f_out.write(f">{record.id} {final_title}\n{final_seq}\n")
                        # ADD TO METADATA LIST (Sync!)
                        metadata_rows.append({
                            "Sequence_ID": record.id,
                            "Full_Description": final_title,
                            "Species": record.annotations.get('organism', 'Unknown'),
                            "Country": country.split(':')[0],
                            "Paper_Title": paper_title,
                            "Isolation_Source": isolation_source,
                            "Authors": authors,
                            "Journal": journal
                        })

                print(f" Progress: {min(start+batch_size, count)}/{count} processed...")
                time.sleep(0.3)
            except Exception as e:
                print(f"❌ Error in batch {start}: {e}")

    return fasta_path, metadata_rows

def append_local_data(ncbi_fasta_path, user_gene, metadata_rows):
    combined_path = ncbi_fasta_path.replace("_raw.fasta", "_combined.fasta")
    all_records = list(SeqIO.parse(ncbi_fasta_path, "fasta"))
    
    folders = ["../raw_data", "../raw_data/Outgroups"]
    for folder in folders:
        if os.path.exists(folder):
            matching_files = [f for f in os.listdir(folder) if f.lower().endswith((".fasta", ".fa")) and user_gene.upper() in f.upper()]
            for f in matching_files:
                local_recs = list(SeqIO.parse(os.path.join(folder, f), "fasta"))
                for record in local_recs:
                    all_records.append(record)
                    metadata_rows.append({
                        "Sequence_ID": record.id,
                        "Full_Description": record.description,
                        "Species": "Local/Outgroup",
                        "Country": "Local",
                        "Paper_Title": "Local Data",
                        "Isolation_Source": "Unknown",
                        "Authors": "User",
                        "Journal": f"File: {f}"
                    })
                print(f"➕ Appended {len(local_recs)} sequences from: {f}")

    SeqIO.write(all_records, combined_path, "fasta")
    return combined_path

# --- 3. THE INTERFACE ---
if __name__ == "__main__":
    print("\n🧬 PHYLOGENETICS PIPELINE")
    org = input("Enter Organism name: ").strip()
    gene = input("Enter Gene (COI, 18S, etc.): ").strip()

    if org and gene:
        base = setup_folders(gene)
        raw_fasta, meta_list = fetch_and_extract(org, gene, base)
        final_fasta = append_local_data(raw_fasta, gene, meta_list)
        
        # Save Sync'd TSV
        meta_path = f"{base}/Metadata/{org}_{gene}_NCBI_info.tsv"
        pd.DataFrame(meta_list).to_csv(meta_path, sep='\t', index=False)
        
        print(f"\n✅ All set! Files are in {base}")